### 1. Imports

In [ ]:
import os
import requests
from typing import TypedDict

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END

### 2. Configurations

In [ ]:
with open(r"") as f:
    openai_api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
CIS_URL = r"https://cloud.flowiseai.com/api/v1/prediction/034ff02d-e9e0-4003-937d-6c37ea84e157"

In [ ]:
with open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-9\key-vault\nvd-database\api.key") as f:
    nvd_api_key = f.read().strip()
NVD_API_KEY = nvd_api_key

In [ ]:
MODEL = "gpt-4.1-mini"
llm = ChatOpenAI(model=MODEL, temperature=0)

### 3. Build the Tools

#### CIS Tool

In [ ]:
def query(payload):
    response = requests.post(CIS_URL, json=payload)
    return response.json()

output = query({
    "question": "Tell me about password policy in windows",
})

In [ ]:
print(output["text"])

In [ ]:
@tool
def ask_cis(question: str) -> str:
    """
    Query the Flowise knowledge base for CIS benchmark for windows related queries and return the response.
    
    Args:
        question: User's question to ask the Flowise chatbot.
    """
    payload = {
        #"query": question
        "question": question
    }

    try:
        #response = requests.post(r"http://127.0.0.1:8000/ask", json=payload, timeout=30)
        response = requests.post(CIS_URL, json=payload, timeout=30)
        response.raise_for_status()

        data = response.json()

        # Most Flowise deployments return {"text": "..."}
        return data.get("text", str(data))

    except requests.RequestException as e:
        return f"Error communicating with Flowise: {e}"

In [ ]:
ask_cis.invoke("Tell me about proper shutdown of windows servers")

#### NVD Tool

In [ ]:
@tool
def lookup_cve(keyword:str)->str:
    """Query the NVD API for CVEs matching a keyword."""
    print("[TOOL]CVE Tool Activated")
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    params = {"keywordSearch": keyword, "resultsPerPage": 3}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        data = r.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return "No CVEs found."
        lines = []
        for v in vulns:
            c = v["cve"]
            lines.append(f"{c['id']}: {c['descriptions'][0]['value'][:180]}")
        return "\n".join(lines)
    except Exception as e:
        return f"CVE lookup failed: {e}"

In [ ]:
lookup_cve.invoke("SMB")

#### IP Config Tool

In [ ]:
import subprocess

@tool
def ipconfig_tool() -> str:
    """
    Returns the Windows network configuration using the ipconfig command.
    """
    try:
        result = subprocess.run(
            ["ipconfig"],
            capture_output=True,
            text=True,
            check=True,
            shell=True
        )

        return result.stdout

    except subprocess.CalledProcessError as e:
        return f"Error executing ipconfig:\n{e.stderr}"

In [ ]:
print(ipconfig_tool.invoke({}))

### 4. Agent Layer

In [ ]:
planner = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a planner. Decide weather RAG and CVE lookup is required"
)

In [ ]:
retrieval_agent = create_agent(
    model=llm,
    tools=[ask_cis],
    system_prompt="Use the CIS policy tool to retrieve CIS benchmark guidance. Use the provided tools mandatorily"
)

In [ ]:
threat_agent = create_agent(
    model=llm,
    tools=[lookup_cve],
    system_prompt="Use CVE lookup tool when threat intelligence is needed. Use the provided tools mandatorily"
)

In [ ]:
validator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Validate the evidence and produce a concise final answer"
)

In [ ]:
response = planner.invoke({ 'messages': 'what is the password requriement for windows?'})

In [ ]:
response

### 5. State

In [ ]:
class CyberState(TypedDict):
    query: str
    plan: str
    rag: str
    cves: str
    final: str

### 6. Nodes

In [ ]:
def planner_node(state: CyberState):
    r = planner.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"plan": str(r)}

In [ ]:
def rag_node(state: CyberState):
    r = retrieval_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"rag": str(r)}

In [ ]:
def cve_node(state: CyberState):
    r = threat_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"cves": str(r)}

In [ ]:
def validator_node(state: CyberState):

    prompt = f"""
User Query:
{state["query"]}

Plan:
{state.get("plan", "")}

RAG:
{state.get("rag", "")}

CVEs:
{state.get("cves", "")}

SCORE:
validation score

Produce the final validated response.
Also, give a score from 0 to 10
"""
    r = validator_agent.invoke({
        "messages":[
            {"role":"user", "content":prompt}
        ]
    })
    return {"final": r}

### 7. Workflow

In [ ]:
graph = StateGraph(CyberState)

graph.add_node("planner", planner_node)
graph.add_node("rag", rag_node)
graph.add_node("cve", cve_node)
graph.add_node("validator", validator_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "rag")
graph.add_edge("rag", "cve")
graph.add_edge("cve", "validator")
graph.add_edge("validator", END)

In [ ]:
app = graph.compile()

### 8. Execute

In [ ]:
result = app.invoke({
    "query": "How can I harden Windows SMB services against ransomware?"
})

In [ ]:
result

In [ ]:
print(result["final"]["messages"][-1].content)

In [ ]:
result = app.invoke({
    "query": "How can I secure windows SMB against latest attacks?"
})

In [ ]:
result

In [ ]:
print(result["final"]["messages"][-1].content)